# 04 Baselines: simple references and Team 3's evaluation protocol

**Purpose.**
1. Score the validation set with simple, untrained rankings, each sorting the candidates by one signal. They are the reference points every trained model is compared with, and they show how strong popularity, relevance and personal history are on their own.
2. Apply the evaluation protocol of the prior baseline (Team 3) to rankings whose quality we know, to see what that protocol measures. Their model itself is re-implemented in notebooks 04b and 04c.

**References** (fixed from training-row evidence in notebook 03, not chosen on validation):

| Reference | Feature | Why |
|---|---|---|
| Random order | constant score | floor |
| BM25 | `bm25` | standard lexical relevance |
| Query-title Jaccard | `title_jaccard` | best lexical signal on training rows |
| Training exposures | `pop_exposures` | strongest non-history popularity on training rows |
| History popularity | `pop_history` | strongest single signal on training rows |
| History popularity, lag 1 day | `pop_history_lag1d` | the deployable version of it |
| Engagement rate | `pop_engaged_rate` | supervised popularity |
| Same product, all history | `pooled_product` | best personal signal |
| Same shop, other products | `pooled_shop_other` | personal evidence that transfers to new products |
| Same product, recent clicks | `click_0_7_product` | strongest history window |
| Same product, query-related history | `qrel_product` | query relevance of history |

**Rules.**
- Complete validation lists, exact expectation over random tie order (`jd.evaluate`). Many candidates tie (e.g. zero popularity); ties are never broken by file order.
- Missing values (`NaN`, unknown metadata) are scored as 0: unknown counts as "no evidence".
- Two populations: all validation searches with an engaged candidate (19,772) and `validation_clean` (18,844, without the clamped final gap), the main population of all later comparisons.
- Nothing is tuned here.

**Runtime:** CPU, about 5 minutes.

In [1]:
# Setup: on Colab, clone the repository (shared code in jdsearch.py) and use Google Drive as the working folder.
import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/repo").is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/liamkunert/BT4222-group17-jdsearch-submission.git", "/content/repo"], check=True)
    sys.path.insert(0, "/content/repo")
else:
    sys.path.insert(0, str(Path.cwd().resolve().parent))
import jdsearch as jd

paths = jd.setup()
jd.require(paths, "processed/search_table.parquet", "processed/split_v1.csv",
           "processed/features_v3_validation.parquet")

Mounted at /content/drive
working folder: /content/drive/MyDrive/BT4222-G17-JDsearch


In [2]:
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.4f}".format
searches = pd.read_parquet(paths.processed / "search_table.parquet")
assignment = pd.read_csv(paths.processed / "split_v1.csv", index_col="session_id")
validation_ids = assignment.index[assignment.split == "validation"]
validation_sizes = searches.loc[validation_ids, "n_candidates"]

## 1. Load the validation features

In [3]:
REFERENCES = {
    "random order": None,
    "BM25": "bm25",
    "query-title Jaccard": "title_jaccard",
    "training exposures": "pop_exposures",
    "history popularity": "pop_history",
    "history popularity, lag 1 day": "pop_history_lag1d",
    "engagement rate": "pop_engaged_rate",
    "same product, all history": "pooled_product",
    "same shop, other products": "pooled_shop_other",
    "same product, recent clicks": "click_0_7_product",
    "same product, query-related history": "qrel_product",
}
columns = ["session_id", "label"] + [f for f in REFERENCES.values() if f]
validation = pd.read_parquet(paths.processed / "features_v3_validation.parquet", columns=columns)
print(f"validation: {validation.session_id.nunique():,} searches, {len(validation):,} candidates")
validation[[f for f in REFERENCES.values() if f]].isna().mean().rename("missing share").to_frame()

validation: 20,000 searches, 1,781,573 candidates


,missing share
bm25,0.0044
title_jaccard,0.0044
pop_exposures,0.0000
pop_history,0.0000
pop_history_lag1d,0.0000
pop_engaged_rate,0.0000
pooled_product,0.0000
pooled_shop_other,0.0141
click_0_7_product,0.0000
qrel_product,0.0000


## 2. Evaluate every reference

In [4]:
per_search = {}
for name, feature in REFERENCES.items():
    validation["score"] = 0.0 if feature is None else validation[feature].fillna(0).astype(float)
    per_search[name] = jd.evaluate(validation, "score", expected_sizes=validation_sizes)
all_ids = per_search["random order"].index
clean_ids = all_ids[~searches.final_gap_clamped.reindex(all_ids).to_numpy()]
print(f"validation searches with an engaged candidate: {len(all_ids):,}; clean: {len(clean_ids):,}")

results = pd.concat({name: jd.summarise(per.loc[clean_ids]).iloc[0] for name, per in per_search.items()}, axis=1).T
results["searches"] = results.searches.astype(int)
results = results.sort_values("ndcg_at_10", ascending=False)
results

validation searches with an engaged candidate: 19,772; clean: 18,844


,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr
history popularity,18844,0.3413,0.3365,0.3463,0.5262,0.3384
"history popularity, lag 1 day",18844,0.3374,0.3327,0.3423,0.5212,0.3352
training exposures,18844,0.3149,0.3102,0.3194,0.5012,0.3112
engagement rate,18844,0.2447,0.2400,0.2492,0.3673,0.2590
"same product, all history",18844,0.1594,0.1560,0.1629,0.2324,0.1927
"same shop, other products",18844,0.1333,0.1306,0.1361,0.2328,0.1487
"same product, recent clicks",18844,0.1330,0.1302,0.1361,0.2062,0.1636
"same product, query-related history",18844,0.1227,0.1201,0.1255,0.1954,0.1509
query-title Jaccard,18844,0.0970,0.0940,0.0997,0.1792,0.1135
BM25,18844,0.0934,0.0906,0.0963,0.1703,0.1106


Paired differences on the same `validation_clean` searches (mean and 95% bootstrap interval of the per-search difference): against random order, and against lagged history popularity, the deployable non-personal reference.

In [5]:
rows = {}
for name in results.index:
    vs_random = jd.paired_difference(per_search[name].loc[clean_ids], per_search["random order"].loc[clean_ids])
    vs_popularity = jd.paired_difference(per_search[name].loc[clean_ids],
                                         per_search["history popularity, lag 1 day"].loc[clean_ids])
    rows[name] = {"vs_random": vs_random[0], "vs_random_low": vs_random[1], "vs_random_high": vs_random[2],
                  "vs_lagged_pop": vs_popularity[0], "vs_lagged_pop_low": vs_popularity[1],
                  "vs_lagged_pop_high": vs_popularity[2]}
pd.DataFrame(rows).T

,vs_random,vs_random_low,vs_random_high,vs_lagged_pop,vs_lagged_pop_low,vs_lagged_pop_high
history popularity,0.2572,0.2525,0.2615,0.0039,0.0030,0.0048
"history popularity, lag 1 day",0.2533,0.2487,0.2575,0.0000,0.0000,0.0000
training exposures,0.2307,0.2262,0.2351,-0.0226,-0.0260,-0.0193
engagement rate,0.1605,0.1561,0.1647,-0.0928,-0.0969,-0.0883
"same product, all history",0.0752,0.0721,0.0784,-0.1780,-0.1834,-0.1728
"same shop, other products",0.0491,0.0467,0.0515,-0.2041,-0.2085,-0.1991
"same product, recent clicks",0.0488,0.0464,0.0516,-0.2044,-0.2093,-0.1993
"same product, query-related history",0.0385,0.0361,0.0410,-0.2148,-0.2195,-0.2098
query-title Jaccard,0.0128,0.0104,0.0152,-0.2405,-0.2454,-0.2347
BM25,0.0092,0.0069,0.0116,-0.2441,-0.2493,-0.2382


## 3. All searches vs the clean population

Searches with a clamped final gap (5.3%) most likely contain actions from during or after the search in their history. Every reference on all searches, on the clean searches and on the clamped ones:

In [6]:
both = pd.DataFrame({
    "all searches": {n: per_search[n].ndcg.mean() for n in REFERENCES},
    "clean searches": {n: per_search[n].ndcg.loc[clean_ids].mean() for n in REFERENCES},
    "clamped searches": {n: per_search[n].ndcg.drop(clean_ids).mean() for n in REFERENCES},
})
both["clean minus all"] = both["clean searches"] - both["all searches"]
for name, feature in REFERENCES.items():
    if name == "random order":
        continue  # logged by notebook 02
    for split_name, ids in [("validation", all_ids), ("validation_clean", clean_ids)]:
        jd.log_result(paths, "04", "references", name, split_name, per_search[name].loc[ids],
                      notes=f"untrained: sort by {feature}; NaN as 0")
both.sort_values("clean searches", ascending=False)

,all searches,clean searches,clamped searches,clean minus all
history popularity,0.3409,0.3413,0.3330,0.0004
"history popularity, lag 1 day",0.3369,0.3374,0.3268,0.0005
training exposures,0.3150,0.3149,0.3178,-0.0001
engagement rate,0.2450,0.2447,0.2517,-0.0003
"same product, all history",0.1820,0.1594,0.6404,-0.0226
"same shop, other products",0.1356,0.1333,0.1824,-0.0023
"same product, recent clicks",0.1554,0.1330,0.6088,-0.0223
"same product, query-related history",0.1361,0.1227,0.4073,-0.0134
query-title Jaccard,0.0973,0.0970,0.1040,-0.0003
BM25,0.0930,0.0934,0.0850,0.0004


## 4. By segment

The random-order floor differs between segments, so each segment is read as the gain over random **within** that segment (`validation_clean`).

In [7]:
info = searches.loc[clean_ids]
segments = {
    "activity in last 7 days": np.where(info.days_since_last_action < 7, "yes", "no"),
    "list size": pd.cut(info.n_candidates, [0, 30, 90, 150, 200], labels=["1-30", "31-90", "91-150", "151-200"]),
    "history length": pd.cut(info.history_len, [0, 30, 100, 300, np.inf], labels=["1-30", "31-100", "101-300", "300+"]),
}
shown = ["history popularity, lag 1 day", "training exposures", "same product, all history",
         "same product, recent clicks", "query-title Jaccard"]
tables = {}
for segment, labels in segments.items():
    by = pd.Series(labels, index=info.index)
    random_mean = per_search["random order"].ndcg.loc[clean_ids].groupby(by, observed=True).mean()
    table = pd.DataFrame({name: per_search[name].ndcg.loc[clean_ids].groupby(by, observed=True).mean() - random_mean
                          for name in shown})
    table.insert(0, "searches", by.value_counts())
    table.insert(1, "random", random_mean)
    tables[segment] = table
pd.concat(tables)

searches  random  \
activity in last 7 days no           2310  0.0816   
                        yes         16534  0.0845   
list size               1-30         1709  0.3321   
                        31-90        9554  0.0672   
                        91-150       5803  0.0536   
                        151-200      1778  0.0371   
history length          1-30         4951  0.0847   
                        31-100       5826  0.0823   
                        101-300      5407  0.0839   
                        300+         2660  0.0878   

                                 history popularity, lag 1 day  \
activity in last 7 days no                              0.2517   
                        yes                             0.2535   
list size               1-30                            0.2086   
                        31-90                           0.3231   
                        91-150                          0.1785   
                        151-200                         0.1647   
history length          1-30                            0.2419   
                        31-100                          0.2513   
                        101-300                         0.2632   
                        300+                            0.2586   

                                 training exposures  \
activity in last 7 days no                   0.2418   
                        yes                  0.2291   
list size               1-30                 0.1875   
                        31-90                0.2909   
                        91-150               0.1676   
                        151-200              0.1549   
history length          1-30                 0.2344   
                        31-100               0.2325   
                        101-300              0.2286   
                        300+                 0.2241   

                                 same product, all history  \
activity in last 7 days no                          0.0268   
                        yes                         0.0820   
list size               1-30                        0.0525   
                        31-90                       0.0895   
                        91-150                      0.0638   
                        151-200                     0.0578   
history length          1-30                        0.0499   
                        31-100                      0.0735   
                        101-300                     0.0896   
                        300+                        0.0970   

                                 same product, recent clicks  \
activity in last 7 days no                            0.0000   
                        yes                           0.0557   
list size               1-30                          0.0314   
                        31-90                         0.0546   
                        91-150                        0.0459   
                        151-200                       0.0442   
history length          1-30                          0.0331   
                        31-100                        0.0491   
                        101-300                       0.0593   
                        300+                          0.0563   

                                 query-title Jaccard  
activity in last 7 days no                    0.0071  
                        yes                   0.0136  
list size               1-30                  0.0434  
                        31-90                 0.0146  
                        91-150                0.0034  
                        151-200               0.0045  
history length          1-30                  0.0091  
                        31-100                0.0078  
                        101-300               0.0188  
                        300+                  0.0183

## Summary of the simple references

- **Popularity dominates:** history popularity reaches NDCG@10 0.341 on `validation_clean`, lagged by one day (deployable) 0.337, training exposures 0.315, the engagement rate 0.245.
- **Best personal reference:** same product in all history, 0.159; recent clicks alone 0.133; shop via other products 0.133; query-related history 0.123.
- **Lexical relevance alone barely beats random** (0.084): Jaccard 0.097, BM25 0.093.
- **Clamped searches:** popularity is unaffected, but the same-product rule reaches 0.640 on the 928 clamped searches (0.159 on the clean ones): their histories contain post-search actions, which is why they are excluded from training and the main evaluation.
- **Segments:** popularity helps with and without recent activity; personal history mostly with activity in the last 7 days and with longer histories.
- **Reference for the trained models:** lagged history popularity, 0.337. Whether personal history adds on top of popularity is the question for notebook 05; single signals cannot answer it.

# Team 3: their evaluation protocol

Team 3 (the prior project baseline on this dataset) reported **HR@10 0.978 and NDCG@10 0.568** for their hybrid NCF + LSTM model. Their evaluation differs from ours in three ways:

1. **Downsampled lists** (`Data Preparation/1. Sampling.ipynb` in their repository): every engaged candidate plus `min(4 x positives, negatives)` randomly drawn non-engaged ones; lists shrink from ~89 to ~9 candidates.
2. **Exponential gains:** `(2^label - 1) / log2(rank + 1)`, so a purchase counts 7, a cart 3, a click 1 (we use 3 / 2 / 1).
3. **Tie handling:** `np.argsort(scores)[::-1]`, ties broken by array order, which in their pipeline is engaged-first.

Here we apply their protocol to **our** validation searches without any model, to see what it does to rankings whose quality we know: random order, history popularity and same-product history. Their NDCG function is copied verbatim (`Models/3.1_Hybrid_NCF_with_LSTM.ipynb`, cell 91). Random draws: 20 random orders per search (ties broken at random, never by array order), seeded per search. All 19,772 validation searches, as in their setting.

In [8]:
def team3_ndcg_at_k(user_preds, user_labels, K=10):
    # Verbatim from Team 3 (Models/3.1_Hybrid_NCF_with_LSTM.ipynb, cell 91).
    topk_indices = np.argsort(user_preds)[::-1][:K]
    topk_labels = np.array(user_labels)[topk_indices]
    dcg = np.sum((2**topk_labels - 1) / np.log2(np.arange(1, len(topk_labels)+1) + 1))
    ideal_labels = np.sort(user_labels)[::-1][:K]
    idcg = np.sum((2**ideal_labels - 1) / np.log2(np.arange(1, len(ideal_labels)+1) + 1))
    return dcg / idcg if idcg > 0 else 0.0


def team3_hit_rate_at_k(user_preds, user_labels, K=10):
    # Verbatim from Team 3 (cell 90).
    topk_indices = np.argsort(user_preds)[::-1][:K]
    topk_labels = np.array(user_labels)[topk_indices]
    return int((topk_labels > 0).any())


def team3_sample(labels, rng):
    # Their sampling: all positives + min(4 x positives, negatives) random negatives.
    pos, neg = np.flatnonzero(labels > 0), np.flatnonzero(labels == 0)
    k = min(4 * len(pos), len(neg))
    return np.concatenate([pos, rng.choice(neg, size=k, replace=False)]) if k else pos


def team3_scores(scores, labels, draws, seed):
    # Mean Team 3 NDCG@10 and HR@10 over random tie orders (scores permuted, then a stable descending sort).
    rng = np.random.default_rng(seed)
    ndcg = hit = 0.0
    for _ in range(draws):
        perm = rng.permutation(len(scores))
        s, l = scores[perm], labels[perm]
        jitter = rng.random(len(s)) * 1e-9 * (np.ptp(s) + 1)  # breaks ties at random, keeps the order otherwise
        ndcg += team3_ndcg_at_k(s + jitter, l)
        hit += team3_hit_rate_at_k(s + jitter, l)
    return ndcg / draws, hit / draws


assert team3_ndcg_at_k(np.array([2.0, 1.0]), np.array([3, 0])) == 1.0
assert np.isclose(team3_ndcg_at_k(np.array([1.0, 2.0]), np.array([3, 0])), 1 / np.log2(3))

In [9]:
TEAM3_RANKINGS = {"random order": None, "history popularity": "pop_history",
                  "same product, all history": "pooled_product"}
DRAWS = 20
frame = validation.sort_values("session_id", kind="stable")
bounds = np.r_[0, np.flatnonzero(np.diff(frame.session_id.to_numpy())) + 1, len(frame)]
labels_all = frame.label.to_numpy()
rows, sampled_parts = [], []
for lo, hi in zip(bounds[:-1], bounds[1:]):
    labels = labels_all[lo:hi]
    if not (labels > 0).any():
        continue  # Team 3 also drops lists that end up empty
    session_id = int(frame.session_id.iat[lo])
    keep = team3_sample(labels, np.random.default_rng([4222, 3, session_id]))
    sampled_parts.append(frame.iloc[lo:hi].iloc[keep])
    row = {"session_id": session_id, "n_full": hi - lo, "n_sampled": len(keep)}
    for name, feature in TEAM3_RANKINGS.items():
        scores = np.zeros(hi - lo) if feature is None else frame[feature].iloc[lo:hi].fillna(0).to_numpy(float)
        row[f"{name}|sampled"] = team3_scores(scores[keep], labels[keep], DRAWS, [4222, 4, session_id])
        row[f"{name}|full"] = team3_scores(scores, labels, DRAWS, [4222, 5, session_id])
    rows.append(row)
team3_runs = pd.DataFrame(rows).set_index("session_id")
sampled = pd.concat(sampled_parts, ignore_index=True)
print(f"searches {len(team3_runs):,}; mean list length full {team3_runs.n_full.mean():.1f}, "
      f"sampled {team3_runs.n_sampled.mean():.1f}; sampled lists with <= 10 candidates "
      f"{(team3_runs.n_sampled <= 10).mean():.1%}")

searches 19,772; mean list length full 89.2, sampled 9.4; sampled lists with <= 10 candidates 80.8%


Four protocols side by side for each ranking: Team 3's protocol (downsampled lists, exponential gains), downsampled lists with our exact metric, full lists with Team 3's formula, and our protocol (full lists, linear gains, exact tie expectation).

In [10]:
table = {}
for name, feature in TEAM3_RANKINGS.items():
    sampled_ndcg = team3_runs[f"{name}|sampled"].map(lambda x: x[0]).mean()
    sampled_hit = team3_runs[f"{name}|sampled"].map(lambda x: x[1]).mean()
    full_ndcg = team3_runs[f"{name}|full"].map(lambda x: x[0]).mean()
    sampled["score"] = 0.0 if feature is None else sampled[feature].fillna(0).astype(float)
    ours_sampled = jd.evaluate(sampled, "score").ndcg.mean()
    table[name] = {"Team 3 protocol: NDCG@10": sampled_ndcg, "Team 3 protocol: HR@10": sampled_hit,
                   "downsampled, our metric": ours_sampled, "full lists, Team 3 formula": full_ndcg,
                   "full lists, our metric": jd.summarise(per_search[name]).ndcg_at_10.iloc[0]}
team3_table = pd.DataFrame(table).T
team3_table.loc["Team 3 model (reported)"] = [0.568, 0.978, np.nan, np.nan, np.nan]
team3_table

,Team 3 protocol: NDCG@10,Team 3 protocol: HR@10,"downsampled, our metric","full lists, Team 3 formula","full lists, our metric"
random order,0.5317,0.9917,0.5351,0.0839,0.0845
history popularity,0.7417,0.9976,0.7463,0.3393,0.3409
"same product, all history",0.5848,0.9933,0.5883,0.1813,0.1820
Team 3 model (reported),0.5680,0.9780,NaN,NaN,NaN


## Summary of Team 3's protocol

- **Their protocol makes random order look strong:** downsampling to all positives plus four negatives each (9.4 instead of 89.2 candidates on average; 81% of the lists have at most 10) gives random order **NDCG@10 0.532 and HR@10 0.992**. Their model reported 0.568 and 0.978, only slightly above random NDCG and a lower hit rate than random.
- **Simple rules beat their reported model on their own protocol:** same product in history and history popularity both score higher.
- **Downsampling is the cause, not the gain formula:** on downsampled lists their exponential formula and our linear exact metric give almost the same values; on full lists they give 0.084 and 0.085 for random order.
- **Consequence:** their reported numbers cannot show a model's ranking quality. Their model is re-implemented and evaluated on complete lists with our protocol in notebooks 04b and 04c.
- Caveat: their reported numbers come from their own test split, not our validation searches; both are uniform samples of the same data.